In [1]:
import numpy as np
import pandas as pd
import glob
from numba import guvectorize
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import os
from datetime import datetime, timezone
import re
import json
import awkward as ak

from dbetto import Props
from legendmeta import LegendMetadata
from lgdo import lh5
import lgdo
from pygama.pargen.dsp_optimize import run_one_dsp
import lgdo.types as lgdo_types
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import matplotlib.patches as mpatches
from collections import defaultdict
from pathlib import Path

import itertools
from tqdm import tqdm
import time

/opt/python-extra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = "/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/latest"
config = Props.read_from(os.path.join(data_path, "config.json"), subst_pathvar=True)["setups"]["l200"]["paths"]
lmeta  = LegendMetadata(config["metadata"])
chmap = ak.Array(lmeta.channelmap(lmeta.dataprod.runinfo.p03.r000.phy.start_key).group("system").geds.values())

In [3]:
detector = "B00035B"
chn = chmap.daq.rawid[chmap.name == detector][0]

# Generate BKG

In [4]:
pet_files = sorted(glob.glob(f"{config['tier_pet']}/phy/l200-p0[346789]-*-phy-tier_pet.lh5"))

# Load in event data. We will get:
# - coincident tells us if an event was a pulser, ged, or spm event
# - geds contains all info related to the geds system such as quality uts, PSD, detector statuses, and energy
# - trigger contains forced triggers and the timestamps for every event


In [ ]:
start = time.time()
data_init = lh5.read_as("/evt", pet_files, field_mask=["coincident", "geds", "trigger"], library="ak")
print("Took", time.time() - start, "s to read")

In [ ]:
is_pos = data_init.geds.quality.is_not_bb_like.is_pos_polarity_bits
is_hig = data_init.geds.quality.is_not_bb_like.is_highly_pos_polarity_bits

extra_qc = (
    ak.fill_none((ak.num(is_pos) == 1) & (ak.firsts(is_pos) == 123), False)
    |
    ak.fill_none((ak.num(is_hig) == 1) & (ak.firsts(is_hig) == 507), False)
)

In [ ]:
data = data_init[
    ~data_init.trigger.is_forced &
    ~data_init.coincident.puls &
    ~data_init.coincident.muon_offline &
    data_init.coincident.geds &
    (data_init.geds.quality.is_bb_like | extra_qc) &
    ak.all(data_init.geds.quality.is_good_channel, axis=-1)
]

In [ ]:
mydet = (ak.num(data.geds.rawid, axis=1) == 1) & (data.geds.rawid[:, 0] == chn)

In [ ]:
events = data[mydet]
ene = events[ak.any((events.geds.rawid == chn), axis=1)].geds.energy

In [ ]:
_, ax = plt.subplots(figsize = (18, 4))
ax.hist(ene, bins = 150, range = (0, 150), histtype = 'step')

ax.set_ylabel("Counts / 1 keV")
ax.set_xlabel("Energy [keV]")
ax.set_xlim(0, 150)
ax.axvline(x=66.7, ls="dashed", color ="red")
ax.axvspan(67+6, 67+16, alpha=0.3)
ax.axvspan(67-16, 67-6, alpha=0.3)
#ax.set_yscale('log')
#ax.legend()
plt.show()

In [ ]:
hg_mask = (
    ak.any(
        (events.geds.rawid == chn) &
        (events.geds.energy > 67 + 6) &
        (events.geds.energy < 67 + 16),
        axis=1
    )
)

lw_mask = (
    ak.any(
        (events.geds.rawid == chn) &
        (events.geds.energy > 67 - 16) &
        (events.geds.energy < 67 - 6),
        axis=1
    )
)

combined_mask = hg_mask | lw_mask

bkg_events = events[combined_mask]

In [ ]:
def get_file_and_idx(timestamp, files):
    """ 
    Takes a provided event timestamp and determines which raw file the event occured in 
    and which event in that raw file corresponds to the event.
    """
    
    file_timestamps = [datetime.strptime(os.path.basename(file).split("-")[4], "%Y%m%dT%H%M%SZ").replace(tzinfo=timezone.utc).timestamp() for file in files]
    
    diffs = np.subtract(file_timestamps, timestamp)
    file_idx = list(diffs).index(diffs[diffs > 0][0]) - 1 # The -1 is necessary to handle an issue with time-zone mismatches between the file and event timestamps
    hit_tstamps = lh5.read('ch1104000/raw/timestamp', files[file_idx])
    hit_idx = np.searchsorted(hit_tstamps, timestamp)

    return file_idx, hit_idx

In [ ]:
raw_files = sorted(glob.glob(f"{config['tier_raw']}/phy/*/*/*.lh5"))

In [ ]:
file_timestamps = np.array([
    datetime.strptime(os.path.basename(f).split("-")[4], "%Y%m%dT%H%M%SZ")
    .replace(tzinfo=timezone.utc)
    .timestamp()
    for f in raw_files
])

In [ ]:
timestamps = bkg_events.trigger.timestamp
raw_ids = bkg_events.geds.rawid
file_idx_all = np.searchsorted(file_timestamps, timestamps, side="right") - 1

In [ ]:
groups = defaultdict(list)

for i in tqdm(range(len(timestamps)), desc="Grouping events"):
    f = file_idx_all[i]
    ch = raw_ids[i][0]
    groups[(f, ch)].append(i)

In [ ]:
outdir = Path(f"data/{detector}")
outdir.mkdir(parents=True, exist_ok=True)

for (file_idx, ch), event_ids in tqdm(groups.items(), desc="Processing files"):

    event_ids = np.asarray(event_ids)

    # event timestamps for this group
    ts = np.asarray(timestamps)[event_ids]

    # read hit timestamps ONCE per file/channel
    hit_tstamps = lh5.read(
        f"ch{ch}/raw/timestamp",
        raw_files[file_idx]
    )

    # vectorized mapping event -> hit index
    hit_idx = np.searchsorted(hit_tstamps, ts)

    # bulk waveform read (THIS is the key speedup)
    event = lh5.read(
        f"ch{ch}/raw",
        raw_files[file_idx],
        idx=hit_idx
    )

    # write once per group (not per event)
    lh5.write(
        event,
        "bkg",
        f"data/{detector}/{detector}_bkg.lh5",
        wo_mode="a"
    )

# Generate signal